In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.6/154.6 kB 3.5 MB/s eta 0:00:0000:01


# Login

In [2]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api)

wandb.init(
    entity="23f2004586-indian-institute-of-technology-madras",
    project="23f2004586-t32026",
    name="model-1-scratch"
)

print("Successfully linked and ready to train!")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004586 (23f2004586-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully linked and ready to train!


# Data Pipelines

In [6]:
import numpy as np
import pandas as pd 
import kagglehub
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import warnings
warnings.filterwarnings("ignore")

In [30]:
train_df = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv")
data_dir = "/kaggle/input/competitions/industrial-surface-defect-detection/public/"
train_img_dir = '/kaggle/input/competitions/industrial-surface-defect-detection/public/train/'

In [31]:
print(f"Total rows in train: {len(train_df)}")
print(train_df.head())

Total rows in train: 45572
     image_id defect_type                                           mask_rle  \
0  4S22I7SA34    defect_a  7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...   
1  4S22I7SA34    defect_b                                                NaN   
2  4S22I7SA34    defect_c                                                NaN   
3  4S22I7SA34    defect_d                                                NaN   
4  DZ899111OE    defect_a                                                NaN   

   width  height  
0    800     128  
1    800     128  
2    800     128  
3    800     128  
4    800     128  


In [32]:
print(train_df.columns)
train_df.head(5)

Index(['image_id', 'defect_type', 'mask_rle', 'width', 'height'], dtype='object')


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [9]:
import torch
from torch.utils.data import Dataset
def rle_decode(mask_rle, shape):
    """
    Decodes a Kaggle-style RLE string into a 2D binary mask.
    mask_rle: string of format 'start length start length...'
    shape: (height, width) of the original image
    """
    # If there is no defect, return an empty mask of all zeros
    if not isinstance(mask_rle, str) or pd.isna(mask_rle):
        return np.zeros(shape[0] * shape[1], dtype=np.uint8).reshape(shape)
    
    s = mask_rle.split()
    # Kaggle RLEs are usually 1-indexed, so we subtract 1
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
        
    return img.reshape(shape)

In [33]:
class DefectDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None):
        self.df = df
        self.image_dir = image_dir
        self.transforms = transforms
        self.image_ids = self.df['image_id'].unique()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # Get the unique image name for this batch
        image_id = self.image_ids[idx]
        
        # Extract all 4 rows corresponding to this single image
        image_rows = self.df[self.df['image_id'] == image_id]
        
        # Load the raw image
        image_path = os.path.join(self.image_dir, f"{image_id}.jpg")
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Generate a mask for each of the 4 defect types
        masks = []
        for _, row in image_rows.iterrows():
            rle_string = row['mask_rle']
            # Decode the mask using the fixed 128x800 dimensions
            single_mask = rle_decode(rle_string, shape=(128, 800))
            masks.append(single_mask)
            
        # Stack the 4 individual masks into one array of shape (128, 800, 4)
        mask = np.stack(masks, axis=-1) 
        
        # Apply optional augmentations
        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        # Convert to PyTorch Tensors
        # Image shape becomes [3, 128, 800]
        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        # Mask shape becomes [4, 128, 800]
        mask = torch.tensor(mask, dtype=torch.float32).permute(2, 0, 1)
        
        return image, mask

In [35]:
sample_dataset = DefectDataset(train_df, image_dir=train_img_dir)
sample_image, sample_mask = sample_dataset[0]
print("Image shape:", sample_image.shape)
print("Mask shape:", sample_mask.shape)  

Image shape: torch.Size([3, 128, 800])
Mask shape: torch.Size([4, 128, 800])


# Model-1: ScratchUNet
### DataLoader

In [36]:
from torch.utils.data import DataLoader
# Creates train DataLoader using batch size of 16
train_loader = DataLoader(sample_dataset, batch_size=16, shuffle=True, num_workers=2)
# Verify batch shapes
batch_images, batch_masks = next(iter(train_loader))
print("Batch Image Shape:", batch_images.shape) # Should be [16, 3, 128, 800]
print("Batch Mask Shape:", batch_masks.shape)   # Should be [16, 4, 128, 800]

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Batch Image Shape: torch.Size([16, 3, 128, 800])
Batch Mask Shape: torch.Size([16, 4, 128, 800])


### The Scratch U-Net

In [37]:
import torch.nn as nn
class ScratchUNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # ENCODER (Extracts features, shrinks image)
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2) #128x800 -- 64x400
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool2d(2) #64x400 -- 32x200
        
        # BOTTLENECK (Deepest representation)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # DECODER (Upsamples back to original size)
        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) 
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1), # 64 in-channels due to skip connection
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2) 
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1), # 32 in-channels due to skip connection
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        
        # FINAL OUTPUT: 4 channels (one for each defect type)
        self.final_conv = nn.Conv2d(16, 4, kernel_size=1)

    def forward(self, x):
        # Encoder Pass
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        # Bottleneck Pass
        b = self.bottleneck(p2)
        
        # Decoder Pass with Skip Connections
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1) 
        d2 = self.dec2(d2)
        
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final_conv(d1)

# Initialize and check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ScratchUNet().to(device)
print(f"Model loaded on: {device}")

Model loaded on: cuda


In [38]:
# Pass our test batch through the untrained model
test_outputs = model(batch_images.to(device))
print("Model Output Shape:", test_outputs.shape) # masks: [16, 4, 128, 800]

Model Output Shape: torch.Size([16, 4, 128, 800])


In [39]:
import torch.optim as optim
from tqdm import tqdm
import wandb

# Initialize a new W&B Run for this specific training session
wandb.init(
    project="23f2004586-t32026", 
    name="Model1-Scratch-UNet",
    config={
        "learning_rate": 1e-3,
        "architecture": "Scratch U-Net",
        "epochs": 3,
        "batch_size": 16
    }
)

# 2. Setup Loss Function and Optimizer
# BCEWithLogitsLoss is required here because our model outputs raw numbers (logits)
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 3 

# 3. The Training Loop
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    
    # progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for images, masks in progress_bar:
        # Move data to the GPU
        images = images.to(device)
        masks = masks.to(device)
        # clear old gradinets
        optimizer.zero_grad()
        
        # Forward pass (predict the masks)
        outputs = model(images)
        
        # Calculate how wrong the predictions were
        loss = criterion(outputs, masks)
        
        # Backward pass (calculate gradients)
        loss.backward()
        
        # Update model weights
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Update the progress bar text
        progress_bar.set_postfix(loss=loss.item())
        
        # 4. Stream metrics live to Weights & Biases!
        wandb.log({"batch_loss": loss.item()})
        
    # Calculate and log the average loss for the entire epoch
    avg_loss = epoch_loss / len(train_loader)
    wandb.log({"epoch_loss": avg_loss, "epoch": epoch + 1})
    print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")

# Close the W&B run when finished
wandb.finish()

Epoch 1/3: 100%|██████████| 713/713 [02:00<00:00,  5.92it/s, loss=0.0159]


Epoch 1 Completed | Average Loss: 0.1440


Epoch 2/3: 100%|██████████| 713/713 [01:40<00:00,  7.08it/s, loss=0.0108]


Epoch 2 Completed | Average Loss: 0.0426


Epoch 3/3: 100%|██████████| 713/713 [01:41<00:00,  7.05it/s, loss=0.0424]

Epoch 3 Completed | Average Loss: 0.0406


batch_loss,██▃▃▃▂▂▁▁▁▁▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▅█
epoch_loss,█▁▁
batch_loss,0.04242
epoch,3
epoch_loss,0.04063


In [64]:
from tqdm import tqdm

# 1. RLE Encoder using the '1 1' valid length hack for empty masks
def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    rle = ' '.join(str(x) for x in runs)
    # Returns a valid non-empty string to satisfy Kaggle's null and length validators
    return rle if rle != "" else "1 1"

# 2. Setup correct paths
test_img_dir = '/kaggle/input/competitions/industrial-surface-defect-detection/public/test/'
sample_sub_path = '/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv'

# Load the exact template provided by the competition
sub_df = pd.read_csv(sample_sub_path)

# Put model in evaluation mode
model.eval()
defect_types = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
predictions = {}

# 3. The Inference Loop
with torch.no_grad(): # Disables gradient tracking to save GPU memory
    test_images = sub_df['image_id'].unique()
    
    for img_id in tqdm(test_images, desc="Generating Predictions"):
        # Look for jpg, fallback to png if necessary
        img_path = os.path.join(test_img_dir, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(test_img_dir, f"{img_id}.png")
            
        image = cv2.imread(img_path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Convert to tensor: shape becomes [1, 3, 128, 800]
        img_tensor = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        img_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Predict
        outputs = model(img_tensor) 
        
        # Apply Sigmoid and threshold at 0.5
        probs = torch.sigmoid(outputs)
        binary_masks = (probs > 0.5).cpu().numpy().astype(np.uint8)[0]
        
        # 4. Encode each channel
        for i, defect in enumerate(defect_types):
            predictions[f"{img_id}_{defect}"] = rle_encode(binary_masks[i])

# 5. Inject predictions into the template dataframe safely
def get_pred(row):
    return predictions.get(f"{row['image_id']}_{row['defect_type']}", "1 1")

sub_df['mask_rle'] = sub_df.apply(get_pred, axis=1)

# 6. Save the final file to Kaggle's working directory
output_path = '/kaggle/working/submission.csv'
sub_df.to_csv(output_path, index=False)
print(f"submission.csv successfully generated and saved to {output_path}!")

Generating Predictions: 100%|██████████| 3000/3000 [00:19<00:00, 153.22it/s]


submission.csv successfully generated and saved to /kaggle/working/submission.csv!


In [66]:
sample = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
display(sample)

,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
...,...,...,...,...
11995,11996,KVYHN07S1K,defect_d,
11996,11997,Q9ZZJDQAIT,defect_a,
11997,11998,Q9ZZJDQAIT,defect_b,
11998,11999,Q9ZZJDQAIT,defect_c,


In [65]:
submission = pd.read_csv("/kaggle/working/submission.csv")
display(submission)

,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,1 1
1,2,WAFNKCRR7Q,defect_b,1 1
2,3,WAFNKCRR7Q,defect_c,1 1
3,4,WAFNKCRR7Q,defect_d,1 1
4,5,YZXKEWSUDD,defect_a,1 1
...,...,...,...,...
11995,11996,KVYHN07S1K,defect_d,1 1
11996,11997,Q9ZZJDQAIT,defect_a,1 1
11997,11998,Q9ZZJDQAIT,defect_b,1 1
11998,11999,Q9ZZJDQAIT,defect_c,1 1


# Milestones
## Milestone 1

In [ ]:
"""
import cv2
# Qn-1
train_df = pd.read_csv('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv')
train_df['has_defect'] = train_df['mask_rle'].notna() & (train_df['mask_rle'].astype(str).str.strip() != '')

class_counts = train_df.groupby('defect_type')['has_defect'].sum()
most_frequent_count = int(class_counts.max())  # defect_c = 4100

img_defect_counts = train_df.groupby('image_id')['has_defect'].sum()
clean_images_count = int((img_defect_counts == 0).sum())

print(most_frequent_count + clean_images_count) 

# Qn-2
train_df['has_defect'] = train_df['mask_rle'].notna() & (train_df['mask_rle'].astype(str).str.strip() != '')

cardinality = train_df.groupby('image_id')['has_defect'].sum()
multi_label_count = int((cardinality > 1).sum())
print(multi_label_count)

# Qn-3
DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
def rle2mask(rle, shape=(128, 800)):
    if pd.isna(rle) or str(rle).strip() == "":
        return np.zeros(shape, dtype=np.uint8)
    s = np.asarray(str(rle).strip().split(), dtype=int)
    starts = s[0::2] - 1
    lengths = s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

wide = train_df.pivot(index='image_id', columns='defect_type', values='mask_rle')
multi_defect_imgs = wide[wide.notna().sum(axis=1) > 1]

overlapping_pixels = 0
for _, row in multi_defect_imgs.iterrows():
    mask_sum = sum(rle2mask(row[c]) for c in DEFECT_CLASSES if pd.notna(row[c]))
    overlapping_pixels += int((mask_sum >= 2).sum())

print(overlapping_pixels) 

# Qn-4
pos_df = train_df[train_df['mask_rle'].notna() & (train_df['mask_rle'].astype(str).str.strip() != '')]

def check_bounds(rle):
    s = np.asarray(str(rle).strip().split(), dtype=int)
    starts, lengths = s[0::2], s[1::2]
    return int((starts + lengths - 1).max()), int((starts + lengths).max())

bounds = pos_df['mask_rle'].apply(check_bounds)
max_inclusive = max(b[0] for b in bounds)
exceed_inclusive = sum(b[0] > 102400 for b in bounds)

max_unadjusted = max(b[1] for b in bounds)
exceed_unadjusted = sum(b[1] > 102400 for b in bounds)

print(f"Inclusive (start + length - 1): {exceed_inclusive} rows, max index {max_inclusive}")
# Output: 0 rows, max index 102400

print(f"Unadjusted (start + length):    {exceed_unadjusted} rows, max index {max_unadjusted}")
# Output: 2 rows, max index 102401

# Qn-5
orig_h, orig_w = 128, 800
new_h, new_w = 256, 256

s_x = new_w / orig_w
s_y = new_h / orig_h
ratio = s_y / s_x

print(f"s_x: {s_x}")
print(f"s_y: {s_y}")
print(f"ratio (s_y / s_x): {ratio}")

# Qn-6
train_csv = glob.glob("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv", recursive=True)[0]
train_df = pd.read_csv(train_csv)
base_dir = os.path.dirname(train_csv)

# Find train images directory
img_dir = [d for d in glob.glob(f"{base_dir}/**", recursive=True) if os.path.isdir(d) and "train" in d.lower()][0]

def rle2mask(rle, shape=(128, 800)):
    s = np.asarray(str(rle).strip().split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, starts + lengths):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

pos_df = train_df[train_df["mask_rle"].notna() & (train_df["mask_rle"].astype(str).str.strip() != "")]
cohens_d = {c: [] for c in ["defect_a", "defect_b", "defect_c", "defect_d"]}
# Sample up to 300 positive rows per class for fast 10-second exact verification
for c in cohens_d.keys():
    sub = pos_df[pos_df["defect_type"] == c].sample(n=min(300, (pos_df["defect_type"] == c).sum()), random_state=42)
    for _, row in sub.iterrows():
        img_path = os.path.join(img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = img.astype(np.float32)
        mask = rle2mask(row["mask_rle"])
        fg = img[mask == 1]
        bg = img[mask == 0]
        n1, n2 = len(fg), len(bg)
        s_pool = np.sqrt(((n1 - 1) * fg.var() + (n2 - 1) * bg.var()) / (n1 + n2 - 2))
        if s_pool > 0:
            cohens_d[c].append((fg.mean() - bg.mean()) / s_pool)

for c, vals in cohens_d.items():
    mean_d = np.mean(vals)
    abs_d = np.mean(np.abs(vals))
    print(f"{c}: signed Cohen's d = {mean_d:.4f} | abs(signed d) = {abs(mean_d):.4f} | mean(|d|) = {abs_d:.4f}")


# Qn-6
import os, glob

train_csv = glob.glob("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv", recursive=True)[0]
train_df = pd.read_csv(train_csv)
base_dir = os.path.dirname(train_csv)

# Find train images directory
img_dir = [d for d in glob.glob(f"{base_dir}/**", recursive=True) if os.path.isdir(d) and "train" in d.lower()][0]

def rle2mask(rle, shape=(128, 800)):
    s = np.asarray(str(rle).strip().split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, starts + lengths):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

pos_df = train_df[train_df["mask_rle"].notna() & (train_df["mask_rle"].astype(str).str.strip() != "")]

cohens_d = {c: [] for c in ["defect_a", "defect_b", "defect_c", "defect_d"]}

# Sample up to 300 positive rows per class for fast 10-second exact verification
for c in cohens_d.keys():
    sub = pos_df[pos_df["defect_type"] == c].sample(n=min(300, (pos_df["defect_type"] == c).sum()), random_state=42)
    for _, row in sub.iterrows():
        img_path = os.path.join(img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = img.astype(np.float32)
        mask = rle2mask(row["mask_rle"])
        fg = img[mask == 1]
        bg = img[mask == 0]
        n1, n2 = len(fg), len(bg)
        s_pool = np.sqrt(((n1 - 1) * fg.var() + (n2 - 1) * bg.var()) / (n1 + n2 - 2))
        if s_pool > 0:
            cohens_d[c].append((fg.mean() - bg.mean()) / s_pool)

for c, vals in cohens_d.items():
    mean_d = np.mean(vals)
    abs_d = np.mean(np.abs(vals))
    print(f"{c}: signed Cohen's d = {mean_d:.4f} | abs(signed d) = {abs(mean_d):.4f} | mean(|d|) = {abs_d:.4f}")

# Qn-7
Y_size = 0

# Model 1: |X1| = 0, |Y| = 0 -> Dice(empty, empty) = 1.0
X1_size = 0
dice_model_1 = 1.0 if (X1_size == 0 and Y_size == 0) else 0.0

# Model 2: |X2| = 2, |Y| = 0 -> Intersection = 0
X2_size = 2
dice_model_2 = (2 * 0) / (X2_size + Y_size)

print(f"{dice_model_1}, {dice_model_2}")  # Output: 1.0, 0.0

# Qn-8
# (a) ForegroundDice (10 positive pairs: 6 at 0.80, 4 at 0.40)
fg_dice = (6 * 0.80 + 4 * 0.40) / 10.0

# (b) FalsePositiveScore (40 empty pairs: 36 with 0 px, 4 with 1024 px out of 102400)
mean_fp_fraction = (36 * 0.0 + 4 * (1024.0 / 102400.0)) / 40.0
fp_score = 1.0 - mean_fp_fraction

# (c) EmptyAccuracy (36 out of 40 empty pairs predicted completely empty)
empty_acc = 36.0 / 40.0

# (d) Final weighted Score
score = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc

print(f"{fg_dice:.4f}, {fp_score:.4f}, {empty_acc:.4f}, {score:.4f}")

# Qn-9

is_empty = train_df['mask_rle'].isna() | (train_df['mask_rle'].astype(str).str.strip() == '')

empty_count = int(is_empty.sum())      # 38371
total_count = len(train_df)            # 45572

# 1. Naive empty-aware mean Dice
score_naive = empty_count / total_count  # 38371 / 45572 = 0.8419863 -> 0.8420

# 2. Official competition score for all-empty predictor
fg_dice = 0.0
fp_score = 1.0
empty_acc = 1.0
score_official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc  # 0.3000

print(f"Empty count: {empty_count}")
print(f"{score_naive:.4f}, {score_official:.4f}")"""

In [ ]:
"""
# Qn-10
def rle2mask(rle, shape=(128, 800)):
    if pd.isna(rle) or str(rle).strip() == "":
        return np.zeros(shape, dtype=np.uint8)
    s = np.asarray(str(rle).strip().split(), dtype=int)
    starts = s[0::2] - 1
    lengths = s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

pos_df = train_df[train_df['mask_rle'].notna() & (train_df['mask_rle'].astype(str).str.strip() != '')]

# 1. Smallest connected component area for defect_b
def_b_rows = pos_df[pos_df['defect_type'] == 'defect_b']
b_areas = []
b_total_pixels = 0
b_pixels_lost_50 = 0

for _, row in def_b_rows.iterrows():
    mask = rle2mask(row['mask_rle'])
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        b_areas.extend(areas)
        b_total_pixels += areas.sum()
        b_pixels_lost_50 += areas[areas < 50].sum()

min_area_b = min(b_areas) if b_areas else 0
lost_pct_b_50 = (b_pixels_lost_50 / b_total_pixels) * 100 if b_total_pixels > 0 else 0

# 3. Percentage of ground-truth pixels lost for defect_a with N = 200
def_a_rows = pos_df[pos_df['defect_type'] == 'defect_a']
a_total_pixels = 0
a_pixels_lost_200 = 0

for _, row in def_a_rows.iterrows():
    mask = rle2mask(row['mask_rle'])
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        a_total_pixels += areas.sum()
        a_pixels_lost_200 += areas[areas < 200].sum()

lost_pct_a_200 = (a_pixels_lost_200 / a_total_pixels) * 100 if a_total_pixels > 0 else 0

print(f"Answer format: {min_area_b}, {lost_pct_b_50:.2f}, {lost_pct_a_200:.2f}")

# Qn-11

train_csv_paths = glob.glob("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv", recursive=True) + glob.glob("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv", recursive=True)
if train_csv_paths:
    train_df = pd.read_csv(train_csv_paths[0])
    
    # Create combination string per image_id
    # First pivot or groupby image_id and collect sorted list of defects present
    pos_df = train_df[train_df['mask_rle'].notna() & (train_df['mask_rle'].astype(str).str.strip() != '')]
    
    # Map each image to its sorted defect list joined by '+'
    image_combos = pos_df.groupby('image_id')['defect_type'].apply(lambda x: '+'.join(sorted(x)))
    
    # All image ids in train_df
    all_image_ids = train_df['image_id'].unique()
    
    combo_dict = {}
    for img_id in all_image_ids:
        if img_id in image_combos:
            combo_dict[img_id] = image_combos[img_id]
        else:
            combo_dict[img_id] = 'clean'
            
    combo_series = pd.Series(combo_dict)
    
    n_combos = combo_series.nunique()
    print(f"Number of unique combination strings: {n_combos}")
    
    # 2. defect_b positive images in validation holdout with exact 20% split per stratum
    # Let's find all images that contain 'defect_b'
    defect_b_images = set(pos_df[pos_df['defect_type'] == 'defect_b']['image_id'])
    
    # For each stratum, validation takes exactly 20% of images in that stratum
    # So defect_b images in validation = sum over strata containing defect_b of (count_defect_b_in_stratum * 0.20)
    # Wait, does every image in a stratum containing defect_b actually contain defect_b? 
    # Yes, because the stratum string itself is defined by the exact set of defects present (e.g. 'defect_b', 'defect_a+defect_b', etc.). 
    # If a stratum string contains 'defect_b', then 100% of the images in that stratum contain defect_b.
    # Therefore, 20% of the images in that stratum will fall into validation.
    
    stratum_counts = combo_series.value_counts()
    b_in_val = 0.0
    for stratum, count in stratum_counts.items():
        if 'defect_b' in stratum.split('+'):
            b_in_val += count * 0.20
            
    print(f"defect_b positive images in validation holdout: {b_in_val} (rounded: {round(b_in_val)})")"""